# Suspect patches: background, flat and near-white images

The Kaggle release of this dataset is clean in the bookkeeping sense (every id has a file, every
file has a label, all images are 96x96x3 uint8). What it *does* contain is a population of
**slide-background patches**: bright, low-texture crops from the edge of the tissue section.
Only a handful are truly degenerate (at least one
is a single flat colour) - the rest are real, legitimate patches with little or no tissue in them.

This notebook exists to answer one question: **do we drop any of them from training?**

Things to keep in mind while looking:

* The label is defined by the **centre 32x32 region only** (outlined in each thumbnail:
  green = labelled tumour, red = labelled normal). The outer ring is context.
* The **test set contains the same kind of patches** and cannot be filtered - all 57,458 must be
  predicted. So a training filter only helps if these images actively mislead the model.
* The metric is ROC AUC over the whole test set, so a few thousand easy negatives are not free to
  throw away: the model has to rank them correctly too.

Everything here reads the memmaps built by `python scripts/repack.py`, so the numbers below cover
**all 220,025 training and 57,458 test images** - no sampling, no extrapolation.

In [ ]:
import json, os, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

DATA = Path(os.environ.get("HCD_DATA", Path.cwd().parent / "data"))
MM    = DATA / "memmap"
CACHE = DATA / "cache"
CACHE.mkdir(parents=True, exist_ok=True)

if not (MM / "train_images.npy").exists():
    raise SystemExit(f"no memmap at {MM} - run `python scripts/repack.py` first")

# mmap_mode="r": nothing is loaded until a row is touched
tr_img = np.load(MM / "train_images.npy", mmap_mode="r")
te_img = np.load(MM / "test_images.npy",  mmap_mode="r")
tr_idx = pd.read_parquet(MM / "train_index.parquet")     # id, row, label
te_idx = pd.read_parquet(MM / "test_index.parquet")      # id, row
STATS  = json.loads((MM / "stats.json").read_text())

CENTER = slice(32, 64)       # the 32x32 region that defines the label

print("train:", tr_img.shape, tr_img.dtype, "| test:", te_img.shape)
print("positives: %.2f%%" % (100 * tr_idx["label"].mean()))
print("exact per-channel mean (0-1):", STATS["mean_0_1"])
print("exact per-channel std  (0-1):", STATS["std_0_1"])

## 1. Per-image statistics over the whole dataset

Vectorised over the memmap in chunks, cached to `data/cache/` as parquet. `white_frac` /
`dark_frac` count pixels that are extreme in **all three channels**, and `sat` (max channel minus
min channel) is a crude "colourfulness" measure - stained tissue has it, blank background does not.

In [ ]:
def scan(images, index, name, chunk=1024):
    cache = CACHE / f"stats_{name}_full.parquet"
    if cache.exists():
        print("cached:", cache.name)
        return pd.read_parquet(cache)

    parts, t0 = [], time.perf_counter()
    for s in range(0, len(images), chunk):
        a  = np.asarray(images[s:s + chunk], dtype=np.float32)      # (B, 96, 96, 3)
        c  = a[:, CENTER, CENTER, :]
        mx = a.max(axis=3)
        mn = a.min(axis=3)
        parts.append(pd.DataFrame({
            "mean":       a.mean(axis=(1, 2, 3)),
            "std":        a.std(axis=(1, 2, 3)),
            "c_mean":     c.mean(axis=(1, 2, 3)),
            "c_std":      c.std(axis=(1, 2, 3)),
            "white_frac": (mn > 230).mean(axis=(1, 2)),
            "dark_frac":  (mx <  25).mean(axis=(1, 2)),
            "sat":        (mx - mn).mean(axis=(1, 2)),
        }))
    df = pd.concat(parts, ignore_index=True)
    df.insert(0, "id", index["id"].to_numpy())
    if "label" in index:
        df["label"] = index["label"].to_numpy()
    df.to_parquet(cache, index=False)
    print(f"scanned {len(df)} images in {time.perf_counter() - t0:.1f}s -> {cache.name}")
    return df

tr = scan(tr_img, tr_idx, "train")
te = scan(te_img, te_idx, "test")
tr[["mean", "std", "c_mean", "c_std", "white_frac", "sat"]].describe().round(2)

## 2. Where do the suspects sit in the distribution?

Train (filled) and test (dashed outline) are overlaid. They should match almost exactly - if they
don't, that is worth knowing before filtering anything.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 3.4))
for ax, col, thr in [(axes[0], "mean", 240), (axes[1], "std", 10),
                     (axes[2], "white_frac", 0.90), (axes[3], "sat", 12)]:
    ax.hist(tr[col], bins=100, alpha=.75, density=True, label="train")
    ax.hist(te[col], bins=100, histtype="step", ls="--", density=True, label="test")
    ax.axvline(thr, color="k", lw=1.2)
    ax.set_title(f"{col}  (line at {thr})")
    ax.set_yticks([])
axes[0].legend()
plt.tight_layout(); plt.show()

# positive rate as a function of brightness: does "bright" really mean "negative"?
bins = pd.cut(tr["mean"], bins=[0, 120, 160, 200, 225, 240, 256])
summary = tr.groupby(bins, observed=True).agg(n=("label", "size"), pos_rate=("label", "mean")).round(3)
summary["pct_of_train"] = (100 * summary["n"] / len(tr)).round(2)
summary

## 3. Look at them

Green box = labelled tumour (centre 32x32 contains tumour tissue), red box = labelled normal.
Thumbnails come straight from the memmap, so this is instant.

In [ ]:
def show_group(df, mask, title, images=tr_img, n=24, cols=8, seed=0):
    sel = df[mask]
    if len(sel) == 0:
        print("nothing matches:", title)
        return
    share = 100 * mask.mean()
    pos   = df.loc[mask, "label"].mean() if "label" in df else float("nan")
    sel   = sel.sample(min(n, len(sel)), random_state=seed)
    rows  = int(np.ceil(len(sel) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.75, rows * 2.05))
    flat = np.ravel(np.atleast_1d(axes))
    for ax, (row_i, r) in zip(flat, sel.iterrows()):
        ax.imshow(images[row_i])                 # df row order == memmap row order
        y = int(r["label"]) if ("label" in r and pd.notna(r["label"])) else -1
        ax.add_patch(Rectangle((31.5, 31.5), 32, 32, fill=False, lw=1.3,
                               color={1: "lime", 0: "red"}.get(y, "yellow")))
        ax.set_title(f"y={y}  mu={r['mean']:.0f} sd={r['std']:.0f}", fontsize=7)
    for ax in flat:
        ax.axis("off")
    fig.suptitle(f"{title}   -   {int(mask.sum())} images ({share:.2f}%), positive rate {pos:.3f}",
                 fontsize=11)
    fig.tight_layout(); plt.show()

show_group(tr, tr["std"] < 10,                                   "A. Essentially flat (std < 10) - the real degenerates")
show_group(tr, tr["mean"] > 240,                                 "B. Near-white (mean > 240)")
show_group(tr, tr["mean"].between(225, 240, inclusive="right"),  "C. Bright but textured (225 < mean <= 240)")
show_group(tr, tr["c_std"] < 10,                                 "D. Flat CENTRE region - the label region is blank")
show_group(tr, tr["mean"].between(110, 190),                     "E. Control: ordinary patches")

### The decisive view: suspects that are labelled **positive**

If bright/blank patches were simply "no tissue, therefore negative", this group would be empty.
It isn't. Look at whether tumour tissue is visible inside the green box - if these labels are
plausible, the images are informative and dropping the group throws away real positives.

In [ ]:
show_group(tr, (tr["label"] == 1) & (tr["mean"] > 225), "F. Positive-labelled bright patches")
show_group(tr, (tr["label"] == 1) & (tr["c_std"] < 15),  "G. Positive-labelled, near-flat centre")

### The same groups in the test set

No labels here - just a check that these patches exist in test at a similar rate, i.e. that they
are part of the task rather than a training-set artefact.

In [ ]:
show_group(te, te["std"] < 10,   "TEST: essentially flat (std < 10)", images=te_img)
show_group(te, te["mean"] > 240, "TEST: near-white (mean > 240)",     images=te_img)

## 4. What would each candidate rule cost?

Exact counts, not estimates. `pos_rate_dropped` high means you are deleting real signal;
`test_imgs` is how many images you would still have to predict after training on none of their kind.

In [ ]:
rules = {
    "std < 10 (flat)":          lambda d: d["std"] < 10,
    "std < 5 (very flat)":      lambda d: d["std"] < 5,
    "mean > 245":               lambda d: d["mean"] > 245,
    "mean > 240":               lambda d: d["mean"] > 240,
    "mean > 225":               lambda d: d["mean"] > 225,
    "white_frac > 0.95":        lambda d: d["white_frac"] > 0.95,
    "c_std < 10 (flat centre)": lambda d: d["c_std"] < 10,
    "sat < 8 (colourless)":     lambda d: d["sat"] < 8,
}
rows = []
for name, f in rules.items():
    m_tr, m_te = f(tr), f(te)
    pos = float(tr.loc[m_tr, "label"].mean()) if m_tr.any() else 0.0
    rows.append(dict(
        rule=name,
        train_imgs=int(m_tr.sum()),
        train_pct=round(100 * m_tr.mean(), 3),
        pos_rate_dropped=round(pos, 3),
        positives_lost=int(tr.loc[m_tr, "label"].sum()),
        test_imgs=int(m_te.sum()),
        test_pct=round(100 * m_te.mean(), 3),
    ))
pd.DataFrame(rows).set_index("rule")